# 01 - Exploratory Data Analysis

ISIC 2019 Skin Lesion Classification (8 classes, 25,331 images)

This notebook covers:
1. Class distribution analysis
2. Sample image visualization
3. Patient metadata analysis (age, sex, anatomical site)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

sns.set_theme(style="whitegrid")

DATA_DIR = "../data/raw"
CLASS_NAMES = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC", "SCC"]

## 1. Load Labels and Metadata

In [ ]:
# Load ground-truth labels (one-hot encoded)
labels_df = pd.read_csv(os.path.join(DATA_DIR, "ISIC_2019_Training_GroundTruth.csv"))
print(f"Total images: {len(labels_df)}")
labels_df.head()

In [ ]:
# Load patient metadata
meta_df = pd.read_csv(os.path.join(DATA_DIR, "ISIC_2019_Training_Metadata.csv"))
print(f"Metadata entries: {len(meta_df)}")
meta_df.head()

## 2. Class Distribution

In [ ]:
# Compute class counts from one-hot labels
class_counts = labels_df[CLASS_NAMES].sum().sort_values(ascending=False)
print("Class distribution:")
print(class_counts)
print(f"\nImbalance ratio (max/min): {class_counts.max() / class_counts.min():.1f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(class_counts.index, class_counts.values, color=sns.color_palette("husl", 8))
axes[0].set_title("Class Distribution (Count)")
axes[0].set_ylabel("Number of Images")
axes[0].tick_params(axis="x", rotation=45)

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct="%1.1f%%",
            colors=sns.color_palette("husl", 8))
axes[1].set_title("Class Distribution (%)")

plt.tight_layout()
plt.show()

## 3. Sample Image Visualization

In [ ]:
# Display 2 sample images per class
fig, axes = plt.subplots(2, 8, figsize=(20, 6))

for col_idx, class_name in enumerate(CLASS_NAMES):
    class_images = labels_df[labels_df[class_name] == 1.0]["image"].tolist()
    for row_idx in range(2):
        if row_idx < len(class_images):
            img_path = os.path.join(DATA_DIR, f"{class_images[row_idx]}.jpg")
            if os.path.exists(img_path):
                img = Image.open(img_path)
                axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].axis("off")
        if row_idx == 0:
            axes[row_idx, col_idx].set_title(class_name, fontsize=12, fontweight="bold")

plt.suptitle("Sample Images per Class", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Metadata Analysis

In [ ]:
# Merge labels with metadata
labels_df["class"] = labels_df[CLASS_NAMES].idxmax(axis=1)
merged = labels_df.merge(meta_df, on="image", how="left")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution
axes[0].hist(merged["age_approx"].dropna(), bins=20, edgecolor="black", alpha=0.7)
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age (approx)")
axes[0].set_ylabel("Count")

# Sex distribution
sex_counts = merged["sex"].value_counts()
axes[1].bar(sex_counts.index, sex_counts.values, color=["steelblue", "coral", "gray"])
axes[1].set_title("Sex Distribution")
axes[1].set_ylabel("Count")

# Anatomical site distribution
site_counts = merged["anatom_site_general"].value_counts()
axes[2].barh(site_counts.index, site_counts.values, color="teal")
axes[2].set_title("Anatomical Site Distribution")
axes[2].set_xlabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Age distribution per class
fig, ax = plt.subplots(figsize=(12, 5))
merged.boxplot(column="age_approx", by="class", ax=ax)
ax.set_title("Age Distribution by Class")
ax.set_xlabel("Class")
ax.set_ylabel("Age (approx)")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# Missing data summary
print("Missing values in metadata:")
print(meta_df.isnull().sum())
print(f"\nMissing percentage:")
print((meta_df.isnull().sum() / len(meta_df) * 100).round(2))